# Feature engineering pipeline

Loads raw data and produces the feature matrix used by all model notebooks.
Applies skill cleaning, fuzzy matching, and semantic title similarity on top of the structured
signals described in PLAN.md. Input: `data/resume.csv`.
Outputs: `outputs/features.csv`, `outputs/title_vecs_cand.npy`, `outputs/title_vecs_job.npy`.

In [1]:
# Imports and load raw CSV from data/
import pandas as pd
import numpy as np
import ast
import re
import os

df = pd.read_csv('../data/resume.csv', encoding='utf-8-sig')
df.columns = [c.lstrip('\ufeff') for c in df.columns]
df = df.rename(columns={
    'educationaL_requirements': 'educational_requirements',
    'experiencere_requirement': 'experience_requirement',
})

# responsibilities.1 — confirmed identical duplicate of responsibilities in EDA (100% row match)
df = df.drop(columns=['responsibilities.1'])

# responsibilities — confirmed job-side in EDA; rename to make that explicit
df = df.rename(columns={'responsibilities': 'job_responsibilities'})

print('Shape:', df.shape)
print('Columns:', df.columns.tolist())

Shape: (9544, 34)
Columns: ['address', 'career_objective', 'skills', 'educational_institution_name', 'degree_names', 'passing_years', 'educational_results', 'result_types', 'major_field_of_studies', 'professional_company_names', 'company_urls', 'start_dates', 'end_dates', 'related_skils_in_job', 'positions', 'locations', 'job_responsibilities', 'extra_curricular_activity_types', 'extra_curricular_organization_names', 'extra_curricular_organization_links', 'role_positions', 'languages', 'proficiency_levels', 'certification_providers', 'certification_skills', 'online_links', 'issue_dates', 'expiry_dates', 'job_position_name', 'educational_requirements', 'experience_requirement', 'age_requirement', 'skills_required', 'matched_score']


## Column drops

Every drop decision is documented in PLAN.md and SUMMARY.md. In brief:

1. **Address / languages / proficiency_levels** — 92–93% null; no job-side counterpart to match against.
2. **Certification cluster** — `certification_skills` is `[None]` in all non-null rows; no usable content.
3. **Extra-curricular / role_positions** — 64% null; negative score correlation (career-stage confounder).
4. **Age requirement** — candidate age not in dataset; direct age discrimination risk.
5. **Educational institution** — no signal without ISCED taxonomy.
6. **Major field of studies** — same: useful only with taxonomy mapping.
7. **Company names / URLs / locations** — no matching signal.
8. **Educational results / result_types** — incomparable grading systems (GPA/4, CGPA/10, %). Bias risk.
9. **Passing years** — messy formats; experience is inferred from start/end dates instead.

In [2]:
# Drop excluded columns; detect and drop 84 empty-profile rows on raw data
DROP_COLS = [
    'address', 'languages', 'proficiency_levels',
    'certification_providers', 'certification_skills',
    'online_links', 'issue_dates', 'expiry_dates',
    'extra_curricular_activity_types', 'extra_curricular_organization_names',
    'extra_curricular_organization_links', 'role_positions',
    'age_requirement',
    'educational_institution_name',
    'professional_company_names', 'company_urls',
    'locations',
    'educational_results', 'result_types',
    'passing_years',
]
df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])
print(f'Shape after column drops: {df.shape}')

Shape after column drops: (9544, 14)


In [3]:
# Drop rows where skills, positions, and start_dates are all null (84 rows)
def is_empty_field(val):
    if val is None: return True
    if isinstance(val, float) and np.isnan(val): return True
    s = str(val).strip()
    return s in ('', '[]', 'None', 'nan', "['']") 

empty_mask = df['positions'].apply(is_empty_field) & df['start_dates'].apply(is_empty_field)
print(f'Empty-profile rows found: {empty_mask.sum()}')
df = df[~empty_mask].reset_index(drop=True)
print(f'Shape after dropping empty profiles: {df.shape}')

Empty-profile rows found: 84
Shape after dropping empty profiles: (9460, 14)


## Parse list-serialised columns

Most columns are stored as Python list strings (e.g. `"['Python', 'SQL']"`).
`skills_required` is an exception — it is newline-separated plain text, confirmed in EDA Q2
(0 of 7,843 non-null rows start with `[`). Do not use `ast.literal_eval` on it.
`related_skils_in_job` is nested lists (e.g. `[['Big Data'], ['Python']]`) — parse then flatten.

In [4]:
# Parse list-serialised columns with ast.literal_eval; split skills_required on newline
def safe_parse(val):
    if pd.isna(val) or str(val).strip() == '':
        return []
    try:
        result = ast.literal_eval(str(val))
        return result if isinstance(result, list) else [result]
    except (ValueError, SyntaxError):
        return [str(val)]

for col in ['skills', 'degree_names', 'major_field_of_studies', 'start_dates', 'end_dates', 'positions']:
    df[col] = df[col].apply(safe_parse)

# skills_required: split on \n only — NOT ast.literal_eval
df['skills_required'] = df['skills_required'].apply(
    lambda v: [s.strip() for s in str(v).split('\n') if s.strip()]
    if pd.notna(v) and str(v).strip() not in ('', 'nan') else []
)

def flatten_nested(lst):
    out = []
    for item in lst:
        if isinstance(item, list):
            out.extend(str(x) for x in item if x is not None)
        elif item is not None:
            out.append(str(item))
    return out

df['related_skils_in_job'] = df['related_skils_in_job'].apply(safe_parse).apply(flatten_nested)

print('Parse check:')
print(f"  skills[0]              : {df['skills'].iloc[0][:3]}")
print(f"  skills_required[2]     : {df['skills_required'].iloc[2][:3]}")
print(f"  related_skils_in_job[0]: {df['related_skils_in_job'].iloc[0][:3]}")
print(f"  positions[0]           : {df['positions'].iloc[0][:3]}")
print(f"  start_dates[0]         : {df['start_dates'].iloc[0][:3]}")
print(f"  degree_names[0]        : {df['degree_names'].iloc[0][:3]}")
print(f"  major_field[0]         : {df['major_field_of_studies'].iloc[0][:3]}")

Parse check:
  skills[0]              : ['Big Data', 'Hadoop', 'Hive']
  skills_required[2]     : ['Brand Promotion', 'Campaign Management', 'Field Supervision']
  related_skils_in_job[0]: ['Big Data']
  positions[0]           : ['Big Data Analyst']
  start_dates[0]         : ['Nov 2019']
  degree_names[0]        : ['B.Tech']
  major_field[0]         : ['Electronics']


## Text normalisation

EDA found 486 of 2,797 unique skill tokens have casing variants (`Python`/`python`,
`SQL`/`Sql`, `JAVA`/`Java`). Fix by lowercasing and deduplicating.
Also clean `career_objective` (0.6% mojibake — `Â` character from UTF-8/Latin-1 misdecode)
and `job_responsibilities` (newline-separated — replace with spaces).

In [5]:
# Lowercase and deduplicate candidate skills; clean job skills_required
before_sample = df['skills'].iloc[0][:3]

# Lowercase + deduplicate (dict.fromkeys preserves insertion order)
df['skills'] = df['skills'].apply(
    lambda lst: list(dict.fromkeys(
        s.lower().strip() for s in lst
        if isinstance(s, str) and s.strip()
    ))
)

df['skills_required'] = df['skills_required'].apply(
    lambda lst: [s.lower().strip() for s in lst if s.strip()]
)

def strip_non_ascii(s):
    if pd.isna(s) or str(s) in ('nan', ''):
        return ''
    return ''.join(c for c in str(s) if ord(c) < 128).strip()

df['career_objective'] = df['career_objective'].apply(strip_non_ascii)

df['job_responsibilities'] = (
    df['job_responsibilities'].fillna('')
    .str.replace('\n', ' ', regex=False)
    .str.strip()
)

df['job_position_name'] = df['job_position_name'].str.strip()

print(f'skills before: {before_sample}')
print(f'skills after : {df["skills"].iloc[0][:3]}')

skills before: ['Big Data', 'Hadoop', 'Hive']
skills after : ['big data', 'hadoop', 'hive']


## Skill cleaning and enrichment

`skills_required` tokens after `\n`-splitting still contain bullet characters,
trailing punctuation, compound entries (`'R or Java'`), and obvious non-skills
(`'good communication'`, `'hard working'`). Clean before computing overlap features.

In [6]:
# Strip bullets/punctuation, split compounds (or/and), remove noise tokens from skills_required
import re

_NOISE_RE = re.compile(
    r'\b(ability|quick learner|hard.?working|good communication|'
    r'fast typing|internet browsing|self.motivated|team player|'
    r'attention to detail|problem.solving|time management)\b',
    re.IGNORECASE
)

def _clean_token(t):
    t = re.sub(r'^[•\-\*\+\#>~]+\s*', '', t)   # strip leading bullets
    t = re.sub(r'[.,;:!?]+$', '', t)                  # strip trailing punctuation
    return t.strip()

def _split_compounds(t):
    """'R or Java' → ['r','java'],  'Python and R' → ['python','r']."""
    for sep in [' or ', ' / ', ' and ']:
        if sep.lower() in t.lower():
            parts = [p.strip() for p in re.split(sep, t, flags=re.IGNORECASE)]
            if all(1 < len(p) < 30 for p in parts):
                return parts
    return [t]

def clean_skills_required(tokens):
    out = []
    for t in tokens:
        t = _clean_token(t)
        if not t or len(t) > 50 or len(t) < 2:
            continue
        if _NOISE_RE.search(t):
            continue
        for part in _split_compounds(t):
            part = _clean_token(part).lower()
            if part and len(part) > 1 and not _NOISE_RE.search(part):
                out.append(part)
    return list(dict.fromkeys(out))   # deduplicate preserving order

# Before/after for a sample of jobs
sample = (df.groupby('job_position_name')['skills_required']
            .first()
            .apply(lambda x: x if isinstance(x, list) else []))
sample = sample[sample.apply(len) > 0].head(6)

print(f'{"job":48s}  {"before":>6}  {"after":>6}  {"example input → output"}')
print('─' * 100)
for job, tokens in sample.items():
    cleaned = clean_skills_required(tokens)
    # Find an interesting example (compound or noise removal)
    example = next((t for t in tokens if ' or ' in t.lower() or ' and ' in t.lower()
                    or _NOISE_RE.search(t) or t.endswith('.') or t.startswith('-')),
                   tokens[0] if tokens else '')
    ex_out = clean_skills_required([example])
    print(f'{job[:48]:48s}  {len(tokens):6d}  {len(cleaned):6d}  '
          f'{repr(example[:30])} → {ex_out}')

df['skills_required'] = df['skills_required'].apply(clean_skills_required)
print(f'\nCleaning applied to all {len(df):,} rows.')

job                                               before   after  example input → output
────────────────────────────────────────────────────────────────────────────────────────────────────
AI Engineer                                            5       5  'r or java' → ['r or java']
Asst. Manager/ Manger (Administrative)                 3       5  '•health safety and environment' → ['health safety', 'environment']
Business Development Executive                         2       0  'fast typing skill' → []
Civil Engineer                                         4       4  'autocad' → ['autocad']
Data Engineer                                          6       6  'azure' → ['azure']
Data Science Engineer                                  6       6  'business analysis' → ['business analysis']



Cleaning applied to all 9,460 rows.


In [7]:
# Recompute skill_coverage and skill_jaccard with cleaned skills_required
# Candidate skills are already lowercased from code-normalise
PREV_COVERAGE_MEAN = 0.0120
PREV_NONZERO       = 480
PREV_NONZERO_PCT   = 0.051

def _coverage(row):
    cand = set(row['skills'])
    job  = set(row['skills_required'])
    return len(cand & job) / len(job) if job else 0.0

def _jaccard(row):
    cand  = set(row['skills'])
    job   = set(row['skills_required'])
    union = cand | job
    return len(cand & job) / len(union) if union else 0.0

df['skill_coverage'] = df.apply(_coverage, axis=1)
df['skill_jaccard']  = df.apply(_jaccard,  axis=1)

new_nonzero     = int((df['skill_jaccard'] > 0).sum())
new_nonzero_pct = (df['skill_jaccard'] > 0).mean()
new_cov_mean    = df['skill_coverage'].mean()

print('After skill cleaning:')
print(f'  skill_coverage mean  : {new_cov_mean:.4f}  (before: {PREV_COVERAGE_MEAN:.4f}  '
      f'delta: {new_cov_mean-PREV_COVERAGE_MEAN:+.4f})')
print(f'  skill_jaccard > 0    : {new_nonzero:,} pairs ({new_nonzero_pct:.1%})'
      f'  (before: {PREV_NONZERO:,} ({PREV_NONZERO_PCT:.1%})'
      f'  delta: {new_nonzero-PREV_NONZERO:+,})')

After skill cleaning:
  skill_coverage mean  : 0.0141  (before: 0.0120  delta: +0.0021)
  skill_jaccard > 0    : 547 pairs (5.8%)  (before: 480 (5.1%)  delta: +67)


## Fuzzy skill matching

Exact token overlap (Jaccard) misses synonyms and near-matches.
`rapidfuzz.token_sort_ratio` handles reordering and abbreviations:
`'machine learning'` ≈ `'ml engineer'`, `'javascript'` ≈ `'js'`.
Threshold 90 is conservative enough to avoid false positives.

In [8]:
# Fuzzy skill matching: rapidfuzz token_sort_ratio at threshold 90
from rapidfuzz import process, fuzz
from scipy.stats import spearmanr as _spr2

threshold = 90  # token_sort_ratio score threshold for considering a fuzzy match

def fuzzy_skill_coverage(candidate_skills, job_skills, threshold=threshold):
    if not job_skills: return 0.0
    matched = 0
    for job_skill in job_skills:
        result = process.extractOne(job_skill, candidate_skills, scorer=fuzz.token_sort_ratio)
        if result and result[1] >= threshold:
            matched += 1
    return matched / len(job_skills)

def fuzzy_skill_jaccard(candidate_skills, job_skills, threshold=threshold):
    if not candidate_skills and not job_skills: return 0.0
    if not job_skills or not candidate_skills: return 0.0
    matched = 0
    for job_skill in job_skills:
        result = process.extractOne(job_skill, candidate_skills, scorer=fuzz.token_sort_ratio)
        if result and result[1] >= threshold:
            matched += 1
    union = len(candidate_skills) + len(job_skills) - matched
    return matched / union if union > 0 else 0.0

print('Computing fuzzy skill features...')
df['fuzzy_skill_coverage'] = df.apply(
    lambda row: fuzzy_skill_coverage(row['skills'], row['skills_required']), axis=1)
df['fuzzy_skill_jaccard']  = df.apply(
    lambda row: fuzzy_skill_jaccard( row['skills'], row['skills_required']), axis=1)

fsc_mean    = df['fuzzy_skill_coverage'].mean()
fsj_nz      = (df['fuzzy_skill_jaccard'] > 0).sum()
fsj_nz_pct  = (df['fuzzy_skill_jaccard'] > 0).mean()
r_fsc, _    = _spr2(df['fuzzy_skill_coverage'], df['matched_score'])

print(f'fuzzy_skill_coverage mean  : {fsc_mean:.4f}  (exact: 0.012)')
print(f'fuzzy_skill_jaccard > 0    : {fsj_nz:,} ({fsj_nz_pct:.1%})  (exact: 480 / 5.1%)')
print(f'Spearman(fuzzy_coverage, matched_score): {r_fsc:.4f}')

# 5 examples where fuzzy matched but exact did not
fuzzy_only = df[(df['fuzzy_skill_coverage'] > 0) & (df['skill_coverage'] == 0)]
print(f'\n5 examples — fuzzy matched, exact did not:')
shown = 0
for _, row in fuzzy_only.iterrows():
    if shown >= 5: break
    for job_skill in row['skills_required']:
        result = process.extractOne(job_skill, row['skills'], scorer=fuzz.token_sort_ratio)
        if result and result[1] >= threshold:
            print(f'  job_req={repr(job_skill):35s} matched={repr(result[0]):35s} '
                  f'sim={result[1]}  score={row["matched_score"]:.3f}')
            shown += 1
            break

Computing fuzzy skill features...


fuzzy_skill_coverage mean  : 0.0149  (exact: 0.012)
fuzzy_skill_jaccard > 0    : 579 (6.1%)  (exact: 480 / 5.1%)
Spearman(fuzzy_coverage, matched_score): 0.1190

5 examples — fuzzy matched, exact did not:
  job_req='autocad'                           matched='auto-cad'                          sim=93.33333333333333  score=0.850
  job_req='market researc'                    matched='market research'                   sim=96.55172413793103  score=0.650
  job_req='elasticsearch'                     matched='elastic search'                    sim=96.2962962962963  score=0.850
  job_req='business analysis'                 matched='business analytics'                sim=91.42857142857143  score=0.727
  job_req='business analysis'                 matched='business analytics'                sim=91.42857142857143  score=0.817


## Rebuild candidate_doc and job_doc

Uses the cleaned `df['skills']` and `df['skills_required']` — same lists as fuzzy matching.
`candidate_doc` = career_objective + skills (cleaned) + positions + related_skils_in_job
`job_doc`        = job_position_name + skills_required (cleaned) + job_responsibilities + educational_requirements

In [9]:
# Rebuild candidate_doc and job_doc using the cleaned, normalised skill lists
# — the same df['skills'] and df['skills_required'] that drive fuzzy matching

def _join(lst):
    return ' '.join(
        str(x) for x in lst
        if x is not None and str(x) not in ('None', 'N/A', 'nan', '')
    )

df['candidate_doc'] = df.apply(lambda row: ' '.join(filter(None, [
    row['career_objective'],
    _join(row['skills']),                   # lowercased + deduped (code-normalise)
    _join(row['positions']),
    _join(row['related_skils_in_job']),
])).strip(), axis=1)

df['job_doc'] = df.apply(lambda row: ' '.join(filter(None, [
    str(row['job_position_name']).strip() if pd.notna(row['job_position_name']) else '',
    ' '.join(row['skills_required']),       # cleaned + compound-split (code-skill-clean)
    str(row['job_responsibilities']).strip()
        if pd.notna(row.get('job_responsibilities', float('nan'))) else '',
    str(row['educational_requirements']).strip()
        if pd.notna(row['educational_requirements']) else '',
])).strip(), axis=1)

clen = df['candidate_doc'].str.len()
jlen = df['job_doc'].str.len()
print(f'candidate_doc: min={clen.min()}  mean={clen.mean():.0f}  max={clen.max()}')
print(f'job_doc      : min={jlen.min()}  mean={jlen.mean():.0f}  max={jlen.max()}')
print(f'\nexample candidate_doc:\n{df["candidate_doc"].iloc[1][:250]}')
print(f'\nexample job_doc:\n{df["job_doc"].iloc[1][:250]}')

candidate_doc: min=104  mean=624  max=3013
job_doc      : min=132  mean=352  max=737

example candidate_doc:
Fresher looking to join as a data analyst and junior data scientist. Experienced in creating meaningful data dashboards and evaluation models. data analysis data analytics business analysis r sas powerbi tableau data visualization business analytics 

example job_doc:
Machine Learning (ML) Engineer Machine Learning Leadership Cross-Functional Collaboration Strategy Development ML/NLP Infrastructure Prototype Transformation ML System Design Algorithm Research Application Development Dataset Selection ML Testing Sta


## Structured features

Four features derived from EDA findings:

1. **`skill_coverage` + `skill_jaccard`** — EDA confirmed near-zero overlap for most pairs (93.8% Jaccard=0,
   structural vocabulary mismatch, not a cleaning issue). Included as weak interpretable features.
2. **`years_experience`** — inferred from `start_dates`/`end_dates`, capped at 25
   (EDA found artefacts beyond 30 years in 224 rows). Null filled with dataset median.
3. **`exp_gap`** — `years_experience` minus parsed `experience_requirement`. EDA confirmed
   consistent format. Null requirement = no constraint = gap of 0.
4. **`edu_match`** — rough ordinal mapping of `degree_names` vs `educational_requirements`.
   Weak signal acknowledged in EDA (Δ=0.021 across all levels), included for interpretability.

In [10]:
# ── skill_coverage + skill_jaccard ───────────────────────────────────────────

def skill_coverage(row):
    cand = set(row['skills'])
    job  = set(row['skills_required'])
    return len(cand & job) / len(job) if job else 0.0

def skill_jaccard(row):
    cand  = set(row['skills'])
    job   = set(row['skills_required'])
    union = cand | job
    return len(cand & job) / len(union) if union else 0.0

df['skill_coverage'] = df.apply(skill_coverage, axis=1)
df['skill_jaccard']  = df.apply(skill_jaccard,  axis=1)

print('skill_coverage:', df['skill_coverage'].describe().round(4).to_dict())
print('skill_jaccard: ', df['skill_jaccard'].describe().round(4).to_dict())


# ── years_experience ──────────────────────────────────────────────────────────

REF_YEAR = 2023

def _extract_year(s):
    if not isinstance(s, str): return None
    m = re.search(r'\b(19|20)\d{2}\b', s)
    return int(m.group()) if m else None

def compute_years_experience(row):
    starts = [_extract_year(str(d)) for d in row['start_dates'] if d is not None]
    ends   = []
    for d in row['end_dates']:
        if d is None: continue
        if isinstance(d, str) and 'till' in d.lower():
            ends.append(REF_YEAR)
        else:
            y = _extract_year(str(d))
            if y: ends.append(y)
    starts = [y for y in starts if y]
    if not starts or not ends: return np.nan
    return float(min(max(max(ends) - min(starts), 0), 25))

df['years_experience'] = df.apply(compute_years_experience, axis=1)
exp_median = df['years_experience'].median()
df['years_experience'] = df['years_experience'].fillna(exp_median).round(1)

print(f'\nyears_experience: median={exp_median:.1f}  mean={df["years_experience"].mean():.1f}  max={df["years_experience"].max()}')


# ── exp_gap ───────────────────────────────────────────────────────────────────

def parse_experience_req(s):
    if not isinstance(s, str): return None
    s = s.lower().strip()
    m = re.search(r'(\d+(?:\.\d+)?)\s*to\s*(\d+(?:\.\d+)?)', s)
    if m: return (float(m.group(1)) + float(m.group(2))) / 2
    m = re.search(r'(\d+(?:\.\d+)?)', s)
    return float(m.group(1)) if m else None

df['_exp_req'] = df['experience_requirement'].apply(parse_experience_req)
df['exp_gap']  = (df['years_experience'] - df['_exp_req']).fillna(0.0).round(1)
df = df.drop(columns=['_exp_req'])

print(f'exp_gap: mean={df["exp_gap"].mean():.2f}  zero_rows={(df["exp_gap"]==0).sum()} (includes null requirement)')


# ── edu_match ─────────────────────────────────────────────────────────────────

def _cand_edu_level(names):
    level = 0
    for n in names:
        if not isinstance(n, str): continue
        n = n.lower()
        if any(x in n for x in ['phd','ph.d','doctor','doctorate']): level = max(level, 3)
        elif any(x in n for x in ['master','m.sc','mba','m.tech','mca','m.e','msc','m.s','pgdm']): level = max(level, 2)
        elif any(x in n for x in ['bachelor','b.sc','b.tech','b.e','b.a','b.com','bca','bba','b.eng','hnd','hsc']): level = max(level, 1)
    return level

def _req_edu_level(s):
    if not isinstance(s, str) or not s.strip(): return 0
    s = s.lower()
    if any(x in s for x in ['phd','ph.d','doctor','doctorate']): return 3
    if any(x in s for x in ['master','m.sc','mba','m.tech','msc','m.s']): return 2
    if any(x in s for x in ['bachelor','b.sc','b.tech','b.e','b.a','graduate','degree']): return 1
    return 0

df['edu_match'] = (
    df['degree_names'].apply(_cand_edu_level) >= df['educational_requirements'].apply(_req_edu_level)
).astype(int)

print(f'edu_match: meets_req={df["edu_match"].sum()}  does_not={(df["edu_match"]==0).sum()}')

# ── directional experience features + skills count ───────────────────────────

df['exp_deficit'] = (df['exp_gap'].clip(upper=0) * -1)   # years short, 0 if over-qualified
df['exp_surplus'] = df['exp_gap'].clip(lower=0)            # years over, 0 if under-qualified

# How many required skills listed — addresses label artefact (null skills_required inflates scores)
df['skills_required_count'] = df['skills_required'].apply(len)

for col in ['exp_deficit', 'exp_surplus', 'skills_required_count']:
    s = df[col]
    print(f'{col:25s}  mean={s.mean():.3f}  nonzero={( s > 0).mean():.1%}')

skill_coverage: {'count': 9460.0, 'mean': 0.0141, 'std': 0.0652, 'min': 0.0, '25%': 0.0, '50%': 0.0, '75%': 0.0, 'max': 1.0}
skill_jaccard:  {'count': 9460.0, 'mean': 0.0031, 'std': 0.0153, 'min': 0.0, '25%': 0.0, '50%': 0.0, '75%': 0.0, 'max': 0.2222}



years_experience: median=4.0  mean=7.0  max=25.0
exp_gap: mean=2.48  zero_rows=1857 (includes null requirement)
edu_match: meets_req=6520  does_not=2940
exp_deficit                mean=1.093  nonzero=33.3%
exp_surplus                mean=3.568  nonzero=47.1%
skills_required_count      mean=3.750  nonzero=78.6%


In [11]:
# Detect fresher/entry-level candidates from career_objective text
import re as _re

_FRESHER_RE = _re.compile(
    r'\b(fresher|fresh graduate|recent graduate|just graduated|entry.level|'
    r'new graduate|looking for.{0,20}first|seeking.{0,20}first)\b',
    flags=_re.IGNORECASE
)

df['is_fresher'] = df['career_objective'].apply(
    lambda s: 1 if isinstance(s, str) and bool(_FRESHER_RE.search(s)) else 0
)

from scipy.stats import spearmanr as _spr3
r_fresh, _ = _spr3(df['is_fresher'], df['matched_score'])
print(f'is_fresher: {df.is_fresher.sum()} candidates ({df.is_fresher.mean():.1%})')
print(f'Mean score  fresher: {df[df.is_fresher==1].matched_score.mean():.3f}')
print(f'Mean score non-fresh: {df[df.is_fresher==0].matched_score.mean():.3f}')
print(f'Spearman with matched_score: {r_fresh:.4f}')

is_fresher: 700 candidates (7.4%)
Mean score  fresher: 0.565
Mean score non-fresh: 0.669
Spearman with matched_score: -0.1622


*Note: the exact skill_coverage mean of 0.012 and Jaccard nonzero rate of 5.1% are pre-cleaning baselines. Fuzzy matching at threshold 90 extends coverage — see code-fuzzy output for updated figures.*

## Skill imputation for jobs with empty skills_required

21.4% of jobs have no `skills_required` — the scoring algorithm inflates all scores for those jobs.
For the 6 affected jobs, impute required skills from the most title-similar jobs using **JobBERT-v3
title embeddings** (not TF-IDF — TF-IDF found wrong neighbors like Mechanical Engineer for ML Engineer).

In [13]:
# Impute skills_required for jobs with empty lists using JobBERT title embeddings
# TF-IDF title matching found wrong neighbors (ML Engineer → Mechanical) —
# semantic embeddings respect the professional meaning of titles.
import numpy as np
from sklearn.preprocessing import normalize as _norm

# Load title embeddings (saved by code-title-sim)
title_vecs = np.load('../outputs/title_vecs_job.npy')  # one per df row, aligned

# Average embedding per unique job title
job_title_embed = {}
for job in df['job_position_name'].unique():
    mask = df['job_position_name'] == job
    job_title_embed[job] = _norm(title_vecs[mask].mean(axis=0, keepdims=True))

jobs_list  = list(job_title_embed.keys())
embed_mat  = np.vstack([job_title_embed[j] for j in jobs_list])
embed_mat  = _norm(embed_mat)

# Group skills by job
job_skills_map = (df.groupby('job_position_name')['skills_required']
                    .apply(lambda g: [s for lst in g for s in lst])
                    .to_dict())
empty_jobs    = [j for j, s in job_skills_map.items() if not s]
nonempty_idx  = [i for i, j in enumerate(jobs_list) if job_skills_map.get(j)]

print(f'Jobs with empty skills_required: {len(empty_jobs)}')

TOPK = 3
for job in empty_jobs:
    idx = jobs_list.index(job)
    sims = (embed_mat[idx:idx+1] @ embed_mat[nonempty_idx].T).flatten()
    top_k = sorted(zip(sims, nonempty_idx), reverse=True)[:TOPK]
    similar = [jobs_list[i] for _, i in top_k]
    imputed = list(dict.fromkeys(s for j in similar for s in job_skills_map[j]))[:10]
    mask = df['job_position_name'] == job
    df.loc[mask, 'skills_required'] = df.loc[mask, 'skills_required'].apply(
        lambda x: imputed if not x else x)
    print(f'  {job[:45]:45s} → {imputed[:3]}')

df['skills_required_count'] = df['skills_required'].apply(len)
print(f'\nskills_required_count: was 0 for {(df.skills_required_count==0).sum()} rows now')
print(f'mean: {df.skills_required_count.mean():.1f}')

Jobs with empty skills_required: 6
  Business Development Executive                → ['corporate marketing', 'facebook ads manager', 'facebook campaign']
  Executive/ Sr. Executive -IT                  → ['having cacc from reputed ca firm', 'internal audit', 'compliance']
  Full Stack Developer (Python,React js)        → ['ansible', 'aws cloud', 'cloud platform']
  Intern (Generative AI Engineering - 2D/3D Ima → ['python', 'r or java', 'tensorflow']
  Machine Learning (ML) Engineer                → ['python', 'r or java', 'tensorflow']
  Senior Software Engineer                      → ['azure', 'big data', 'data analytics']

skills_required_count: was 0 for 0 rows now
mean: 5.8


## Save outputs

Final feature matrix saved to `outputs/features.csv`.
Columns: all remaining raw columns + `candidate_doc` + `job_doc`
+ `skill_coverage` + `skill_jaccard` + `years_experience` + `exp_gap` + `edu_match` + `matched_score`.

In [ ]:
# Encode candidate positions and job title with JobBERT-v3; compute cosine similarity
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize as _norm
from scipy.stats import spearmanr as _spr
import torch
import time

# Encode candidate positions vs job title for semantic title similarity
positions_text = df['positions'].apply(
    lambda lst: ' '.join(str(p) for p in lst
                         if p is not None and str(p) not in ('None','N/A','nan'))
).tolist()
job_title_text = df['job_position_name'].str.strip().fillna('').tolist()

TITLE_CAND_PATH = '../outputs/title_vecs_cand.npy'
TITLE_JOB_PATH  = '../outputs/title_vecs_job.npy'

if os.path.exists(TITLE_CAND_PATH) and os.path.exists(TITLE_JOB_PATH):
    title_vecs_cand = np.load(TITLE_CAND_PATH)
    title_vecs_job  = np.load(TITLE_JOB_PATH)
    print('Title vectors loaded from disk.')
else:
    # TechWolf/JobBERT-v3 — HF fallback if pooling config incompatible
    def _encode_texts(model_name, texts, batch_size=64):
        try:
            m = SentenceTransformer(model_name)
            return m.encode(texts, batch_size=batch_size,
                            show_progress_bar=True, convert_to_numpy=True)
        except TypeError:
            from transformers import AutoTokenizer, AutoModel
            tok = AutoTokenizer.from_pretrained(model_name)
            mdl = AutoModel.from_pretrained(model_name)
            mdl.eval()
            out = []
            for i in range(0, len(texts), batch_size):
                batch = texts[i:i+batch_size]
                enc = tok(batch, padding=True, truncation=True,
                          max_length=128, return_tensors='pt')
                with torch.no_grad():
                    hidden = mdl(**enc).last_hidden_state
                mask = enc['attention_mask'].unsqueeze(-1).float()
                vecs = (hidden * mask).sum(1) / mask.sum(1)
                out.append(vecs.detach().numpy())
            import numpy as _np
            return _np.vstack(out)

    print('Encoding candidate positions and job titles with TechWolf/JobBERT-v3...')
    t0 = time.time()
    title_vecs_cand = _encode_texts('TechWolf/JobBERT-v3', positions_text)
    title_vecs_job  = _encode_texts('TechWolf/JobBERT-v3', job_title_text)
    os.makedirs('../outputs', exist_ok=True)
    np.save(TITLE_CAND_PATH, title_vecs_cand)
    np.save(TITLE_JOB_PATH,  title_vecs_job)
    print(f'Encoded in {time.time()-t0:.1f}s, saved to disk.')

cn = _norm(title_vecs_cand, norm='l2')
jn = _norm(title_vecs_job,  norm='l2')
title_sim = (cn * jn).sum(axis=1)
df['title_semantic_sim'] = title_sim

r_ts, _ = _spr(title_sim, df['matched_score'])
print(f'title_semantic_sim: mean={title_sim.mean():.4f}  std={title_sim.std():.4f}  '
      f'min={title_sim.min():.4f}  max={title_sim.max():.4f}')
print(f'Spearman with matched_score: {r_ts:.4f}')

In [ ]:
# Overwrite outputs/features.csv with all 27 columns
os.makedirs('../outputs', exist_ok=True)
df.to_csv('../outputs/features.csv', index=False)

print(f'Saved: ../outputs/features.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')